In [17]:
from typing_extensions import TypedDict, Literal, Annotated, Dict 
from langgraph.graph import StateGraph, START, END
from langchain_core.tools import tool
from langchain.agents import create_agent
from langgraph.types import Command
from langchain_ollama import ChatOllama
from langchain_core.messages import AIMessage
from pathlib import Path
import os
import json
import subprocess

In [20]:
TOOLS = [
    {"name": "bash",       "description": "Run a shell command."},
    {"name": "read_file",  "description": "Read file contents."},
    {"name": "write_file", "description": "Write content to file."},
    {"name": "edit_file",  "description": "Replace text in file once."},
    {"name": "glob",       "description": "Find files by pattern."},
]

SYSTEM = f"""You are a coding agent at {os.getcwd()}. Use the tools available to solve tasks. Act, don't explain. If a task has been completed, output STOP.
The following tools are available: {str(TOOLS)}

"""

MAX_ITER = 5

WORKDIR = Path.cwd()

def safe_path(p: str) -> Path:
    path = (WORKDIR / p).resolve()
    if not path.is_relative_to(WORKDIR):
        raise ValueError(f"Path escapes workspace: {p}")
    return path

@tool
def run_bash(command : str) -> str:
    """
    Run a bash command and return the output.
    """
    dangerous_commands = ["rm", "sudo", "shutdown", "reboot"]
    if any(dangerous_command in command for dangerous_command in dangerous_commands):
        return "Error: Command contains potentially dangerous operations. Aborting execution."
    try:
        result = subprocess.run(command, shell=True, cwd=os.getcwd(), text=True, capture_output=True, timeout=120)
        out = (result.stdout + result.stderr).strip()
        return out[:10000] if out else "(No output)"
    except subprocess.CalledProcessError as e:
        return f"Error: {e.stderr}"

@tool 
def run_read(path, limit=None):
    """
    Read the contents of a file.
    """
    lines = safe_path(path).read_text().splitlines()
    if limit:
        lines = lines[:limit]
    return "\n".join(lines)

@tool
def run_write(path, content):
    """
    Write content to a file.
    """
    safe_path(path).write_text(content)
    return f"Wrote {len(content)} bytes to {path}"

@tool
def run_edit(path, old_text, new_text):
    """
    Edit a file by replacing the first occurrence of old_text with new_text.
    """
    text = safe_path(path).read_text()
    if old_text not in text:
        return "Error: text not found"
    safe_path(path).write_text(text.replace(old_text, new_text, 1))
    return f"Edited {path}"

@tool
def run_glob(pattern):
    """
    Glob for files matching a pattern.
    """
    import glob as g
    return "\n".join(g.glob(pattern, root_dir=WORKDIR))

def messages_reducer(left : list, right : list) -> list:
    return left + right

class State(TypedDict):
    messages : Annotated[list[Dict[str, str]], messages_reducer]
    max_iterations : int

def call_function(state: State) -> Command:
    all_messages = state.get("messages", [])
    #print("All Messages:", all_messages)
    if state.get("max_iterations", 0) <= 0 or not all_messages:
        if not all_messages:
            mess = f"No messages to process. Stopping execution."
        else:
            mess = f"Reached maximum iterations. Stopping execution."
        return Command(
            update={"messages": [AIMessage(content=mess)]},
            goto=END
        )
    
    last_message = all_messages[-1]

    content = last_message.content if hasattr(last_message, "content") else last_message.get("content", "")

    if "stop" in content.strip().lower():
        return Command(
            update={"messages": [AIMessage(content="Received STOP signal. Ending execution.")]},
            goto=END
        )
    elif "stop" not in content.strip().lower() and state.get("max_iterations", 0) > 0:        
        model = ChatOllama(model="gemma4", temperature=0.1)
        agent = create_agent(model=model, system_prompt=SYSTEM, tools=[run_bash, run_read, run_write, run_edit, run_glob])
        result = agent.invoke(state)
        #print(result)
        return Command(
            update={
                "messages": result.get("messages", []),
                "max_iterations": state.get("max_iterations", 0) - 1
            },
            goto="call_function"
        )
    else:
        return Command(
            goto=END
        )
  
builder = StateGraph(State)
builder.add_node("call_function", call_function)
builder.add_edge(START, "call_function")

# builder.add_edge(START, "call_function")
# builder.add_edge("call_function", END)

graph = builder.compile()

initial_state = {
    "messages": [
        {"role": "user", "content": 'Create a file called test.py that prints "hello", then read it back'}
    ],
    "max_iterations": MAX_ITER
}

result2 = graph.invoke(initial_state)
print("Final State:", result2)

Final State: {'messages': [{'role': 'user', 'content': 'Create a file called test.py that prints "hello", then read it back'}, HumanMessage(content='Create a file called test.py that prints "hello", then read it back', additional_kwargs={}, response_metadata={}, id='69737a89-6e7d-43fc-97d2-833696f1dc3c'), AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'gemma4', 'created_at': '2026-05-23T20:41:05.654987Z', 'done': True, 'done_reason': 'stop', 'total_duration': 109111054200, 'load_duration': 27874095000, 'prompt_eval_count': 426, 'prompt_eval_duration': 54042101000, 'eval_count': 128, 'eval_duration': 25638138600, 'logprobs': None, 'model_name': 'gemma4', 'model_provider': 'ollama'}, id='lc_run--019e5690-473b-79d0-bd13-4cb0a520aa8e-0', tool_calls=[{'name': 'run_write', 'args': {'content': 'print("hello")', 'path': 'test.py'}, 'id': 'e2e38553-645d-448b-a121-0b1ec9866d0e', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 426, 'output_to